# Análisis de video y estado del tráfico

Workflow de inferencia de VAAET ML 4.3.0. Ejecuta bundles piloto o aprobados, persiste las 19 features y mantiene revisión HITL append-only.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(f"{REPO_ROOT}[vision,training,visualization,database]")
else:
    install_command.extend(["-e", f"{REPO_ROOT}[vision,training,visualization,database]"])
subprocess.check_call(install_command)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab must load the installed wheel, not repository path: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"Local editable install has unexpected origin: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: no broken requirements found")
else:
    print("⚠️ pip check detected conflicts in the managed notebook runtime:")
    print(pip_check_output or "No diagnostic output was returned")
    print("ℹ️ Continuing because workflow imports are validated explicitly below.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf
import ultralytics

from vaaet.artifacts import MANIFEST_FILE, validate_manifest
from vaaet.data.database import DatabaseProfile, database_engine, get_optional_database_settings, inspect_database, load_reviewer_id
from vaaet.data.persistence import persist_classified_telemetry
from vaaet.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet.data.review import build_review_widget, export_offline_review_package, load_review_queue, persist_human_validation, select_review_queue
from vaaet.inference.traffic_state import classify_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.settings import DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABEL_MAP_PATH, MODEL_DIR, MODEL_PATH, RANDOM_SEED, SCALER_PATH, STATE_LABELS
from vaaet.vision.analysis import TrafficStatePrediction, analyze_video

configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Package: {VAAET_PACKAGE_FILE}")
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f"✅ inference workflow ready | root={REPO_ROOT} | commit={GIT_COMMIT}")

_model_dir_abs = REPO_ROOT / MODEL_DIR
_model_dir_abs.mkdir(parents=True, exist_ok=True)
_ARTIFACT_NAMES = [Path(MODEL_PATH).name, Path(SCALER_PATH).name, Path(LABEL_MAP_PATH).name, MANIFEST_FILE]

def _bundle_paths(directory: Path) -> dict[str, Path]:
    return {name: directory / name for name in _ARTIFACT_NAMES}

paths = _bundle_paths(_model_dir_abs)
source = "local"
if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_paths = _bundle_paths(Path("/content/drive") / DRIVE_ARTIFACT_DIR)
        if all(path.is_file() for path in drive_paths.values()):
            for name, path in drive_paths.items():
                shutil.copy2(path, paths[name])
            source = "Google Drive"
    except Exception as exc:
        print(f"Drive unavailable: {exc}")

if not all(path.is_file() for path in paths.values()) and IN_COLAB:
    from google.colab import files

    print("Upload the complete four-file model bundle:", _ARTIFACT_NAMES)
    for name, content in files.upload().items():
        if name in _ARTIFACT_NAMES:
            paths[name].write_bytes(content)
    source = "upload"

if all(path.is_file() for path in paths.values()):
    manifest = validate_manifest(_model_dir_abs)
    DEPLOYMENT_STAGE = manifest["training_lifecycle"]["deployment_stage"]
    MODEL_INPUT_POLICY = manifest["training_lifecycle"]["input_policy"]
    ALLOW_PILOT_BUNDLE = True  # Explicit opt-in for the seed-bootstrap pilot.
    ALLOW_EXPERIMENTAL_BUNDLE = False  # Candidate bundles remain offline-only unless explicitly enabled.
    if DEPLOYMENT_STAGE == "pilot" and not ALLOW_PILOT_BUNDLE:
        raise RuntimeError("Pilot bundle loading is disabled. Set ALLOW_PILOT_BUNDLE=True explicitly.")
    if DEPLOYMENT_STAGE == "candidate" and not ALLOW_EXPERIMENTAL_BUNDLE:
        raise RuntimeError("Candidate bundle is not approved. Enable it only for explicit offline evaluation.")
    model = tf.keras.models.load_model(paths[Path(MODEL_PATH).name])
    scaler = joblib.load(paths[Path(SCALER_PATH).name])
    label_mapping = joblib.load(paths[Path(LABEL_MAP_PATH).name])
    if dict(label_mapping) != dict(STATE_LABELS):
        raise RuntimeError("Bundle label_mapping.joblib does not match the four public states.")
    if int(model.output_shape[-1]) != 3:
        raise RuntimeError("Bundle MLP must expose exactly three stable-state outputs.")
    if int(getattr(scaler, "n_features_in_", -1)) != len(FEATURE_COLS):
        raise RuntimeError("Bundle scaler does not match the 19-feature contract.")
    print(f"✅ Valid {DEPLOYMENT_STAGE.upper()} bundle loaded from {source} | input_policy={MODEL_INPUT_POLICY}")
else:
    missing = [name for name, path in paths.items() if not path.is_file()]
    raise FileNotFoundError(f"Incomplete model bundle; missing: {missing}")


## 1. Seleccionar el clip

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Clip: {VIDEO_PATH or 'not selected'}")

## 2. Análisis anotado e inferencia

In [ ]:
def prediction_provider(raw: pd.DataFrame) -> TrafficStatePrediction | None:
    try:
        classified = classify_raw_telemetry(
            raw,
            model,
            scaler,
            label_mapping=label_mapping,
            feature_cols=FEATURE_COLS,
            model_version=manifest["model_version"],
            input_policy=MODEL_INPUT_POLICY,
            inference_mode="stable",
            decision_policy=manifest["decision_policy"],
        )
    except ValueError:
        return None
    if classified.empty:
        return None
    latest = classified.iloc[-1]
    return TrafficStatePrediction(
        state=int(latest["traffic_state"]),
        label=str(latest["state_label"]),
        confidence=float(latest["confidence"]),
        evidence=float(latest.get("accident_evidence_score", 0.0)),
    )

if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Select or upload a valid MP4 clip before continuing")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_analyzed.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.INFERENCE, git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem, model_version=manifest["model_version"])
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    analysis_result = analyze_video(VIDEO_PATH, OUTPUT_VIDEO, prediction_provider=prediction_provider)
    _run.set_output_rows(len(analysis_result.telemetry))
LOCAL_INFERENCE_RUN_ID = str(_run.id)
df_telemetry = analysis_result.telemetry
df_classified = None
INFERENCE_PIPELINE_RUN_ID = None
OFFLINE_VALIDATIONS = []
if df_telemetry.empty:
    print("ℹ️ The clip was analyzed successfully but contains no complete 60-second window.")
    print("   Feature engineering, classification, persistence, and HITL review were skipped.")
    print(f"   Processed: {analysis_result.processed_duration_seconds:.1f}s | minimum required: 60.0s")
else:
    df_classified = classify_raw_telemetry(
        df_telemetry,
        model,
        scaler,
        label_mapping=label_mapping,
        feature_cols=FEATURE_COLS,
        model_version=manifest["model_version"],
        input_policy=MODEL_INPUT_POLICY,
        inference_mode="stable",
        decision_policy=manifest["decision_policy"],
    )
    display(df_classified)
    print(f"✅ Complete minutes: {analysis_result.complete_minutes} | discarded tail: {analysis_result.discarded_partial_seconds:.1f}s")
print(f"✅ Annotated video: {analysis_result.video_path}")

if IN_COLAB:
    from google.colab import files

    files.download(str(analysis_result.video_path))

# Cell 3 — Feature Engineering + Classification
#
# This cell always recomputes classification from df_telemetry using the
# shared vaaet.inference.traffic_state module so that accident gating and contracts stay
# aligned with the training workflow.

try:
    if df_telemetry is not None and not df_telemetry.empty:
        df_classified = classify_raw_telemetry(
            df_telemetry,
            model,
            scaler,
            label_mapping=label_mapping,
            model_version=manifest["model_version"],
            decision_policy=manifest["decision_policy"],
            input_policy=MODEL_INPUT_POLICY,
        )
        if df_classified.empty:
            print("⚠️ Insufficient telemetry for classification")
        else:
            print("✅ Classification complete:")
            for code in sorted(df_classified["traffic_state"].unique()):
                count = int((df_classified["traffic_state"] == code).sum())
                label = STATE_LABELS.get(int(code), "Unknown")
                print(f"   {label:>10}: {count} records")
            if "accident_gate_applied" in df_classified.columns:
                gated = int(df_classified["accident_gate_applied"].sum())
                print(f"   Automatic Accident states: {gated} (must always be zero)")
            candidates = int(df_classified.get("accident_alert_started", pd.Series(False, index=df_classified.index)).sum())
            print(f"   Possible-incident candidates (state remains Congested): {candidates}")
    else:
        print("ℹ️ No complete 60-second telemetry window is available; classification remains skipped.")
        df_classified = None
except NameError:
    print("⚠️ df_telemetry not defined — run Cell 2 or Cell 2b first")
    df_classified = None
except Exception as e:
    print(f"🔴 Classification error: {e}")
    df_classified = None


In [ ]:
# Cell 4 — Persist Results to Database (Optional)

PERSIST_TO_DATABASE = False
inference_settings = (
    get_optional_database_settings(DatabaseProfile.INFERENCE) if PERSIST_TO_DATABASE else None
)

if inference_settings is not None:
    try:
        if df_classified is not None and not df_classified.empty:
            with database_engine(inference_settings) as db_engine:
                health = inspect_database(db_engine, DatabaseProfile.INFERENCE)
                print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
                persisted = persist_classified_telemetry(
                    df_classified, engine=db_engine, model_version=manifest["model_version"]
                )
            INFERENCE_PIPELINE_RUN_ID = persisted.pipeline_run_id
            print(f"✅ Persistence completed: {persisted.telemetry_rows} feature rows | {persisted.classification_rows} predictions | run={INFERENCE_PIPELINE_RUN_ID}")
        else:
            print("ℹ️ Persistence skipped: no classified complete-minute rows are available.")
    except NameError:
        print("⚠️ Run Cells 2-3 first, then re-run this cell")
    except Exception as error:
        print(f"🔴 Persistence failed ({type(error).__name__}). Verify migration, TLS and role grants.")
        print("   Classified data is available in-memory (df_classified)")
else:
    print("ℹ️ Database persistence disabled or profile unavailable; data stays in df_classified.")
    print("   Set PERSIST_TO_DATABASE=True and configure the inference profile to enable.")


## Feedback humano y reentrenamiento

Este workflow sólo persiste telemetría, predicciones y correcciones humanas. El reentrenamiento se ejecuta exclusivamente en `train_traffic_state_classifier.ipynb`, con particiones agrupadas, holdout temporal y promoción manual.


In [ ]:
# Cell 5 — Explicit human review (optional, after inference)
ENABLE_HUMAN_REVIEW = False
REVIEW_MODE = "priority"  # priority or all

if ENABLE_HUMAN_REVIEW:
    reviewer_id = load_reviewer_id()
    review_settings = get_optional_database_settings(DatabaseProfile.REVIEW)
    if review_settings is not None and INFERENCE_PIPELINE_RUN_ID is not None:
        review_queue = load_review_queue(
            settings=review_settings, pipeline_run_id=INFERENCE_PIPELINE_RUN_ID, mode=REVIEW_MODE
        )
        build_review_widget(
            review_queue, reviewer_id=reviewer_id,
            on_submit=lambda decision: persist_human_validation(decision, settings=review_settings),
        )
    elif df_classified is not None and not df_classified.empty:
        print("⚠️ Review database profile unavailable; showing a local queue only.")
        local_queue = df_classified.copy().reset_index(drop=True)
        local_queue['prediction_id'] = local_queue.index + 1
        local_queue = select_review_queue(local_queue, mode=REVIEW_MODE)
        OFFLINE_VALIDATIONS = []
        build_review_widget(local_queue, reviewer_id=reviewer_id, on_submit=OFFLINE_VALIDATIONS.append)
        def export_completed_offline_review(output_path=REPO_ROOT / 'data/processed/vaaet-training-dataset-v1.zip'):
            package = export_offline_review_package(
                output_path, classified=local_queue, validations=OFFLINE_VALIDATIONS
            )
            print(f"✅ HITL package exported: {package}")
            return package
        print("After completing the queue, run export_completed_offline_review().")
    else:
        print("ℹ️ HITL review skipped: no classified complete-minute rows are available.")
else:
    print("ℹ️ Human review disabled. Set ENABLE_HUMAN_REVIEW=True after processing the clip.")


## Visualization

Summary dashboard showing traffic state distribution, speed timeline, and
classification confidence. Only runs after Cells 2-3 have been executed.

In [ ]:
# Cell 6 — Visualization Dashboard

import matplotlib.pyplot as plt

def show_dashboard(df: pd.DataFrame) -> None:
    """Display a 5-panel summary dashboard for the classified telemetry.

    Panels:
      1. Traffic state distribution (bar)
      2. Average speed over time (line)
      3. Classification confidence histogram
      4. Vehicle counts by type (stacked area)
      5. Speed vs. total vehicles scatter

    Args:
        df: Classified DataFrame with traffic_state, avg_speed, confidence,
            and per-type count columns.
    """
    fig = plt.figure(figsize=(20, 10))
    colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
    type_colors = {
        "car": "#3498db", "truck": "#e67e22", "bus": "#e74c3c",
        "motorcycle": "#2ecc71", "bicycle": "#9b59b6",
    }

    # Panel 1: State distribution
    ax1 = fig.add_subplot(2, 3, 1)
    dist = df["traffic_state"].value_counts().sort_index()
    state_names = [label_mapping.get(c, f"State {c}") for c in sorted(dist.index)]
    state_colors = [colors[c] for c in sorted(dist.index)]
    ax1.bar(state_names, dist.values, color=state_colors)
    ax1.set_title("Traffic State Distribution")
    ax1.set_ylabel("Records")

    # Panel 2: Speed timeline
    ax2 = fig.add_subplot(2, 3, 2)
    x_axis = range(len(df))
    ax2.plot(x_axis, df["avg_speed"], color="#3498db", linewidth=1.5, label="Avg Speed")
    if "speed_variance" in df.columns:
        ax2.fill_between(
            x_axis,
            df["avg_speed"] - df["speed_variance"].clip(lower=0).pow(0.5),
            df["avg_speed"] + df["speed_variance"].clip(lower=0).pow(0.5),
            alpha=0.2, color="#3498db", label="±1 σ",
        )
    ax2.set_title("Average Speed Over Time")
    ax2.set_xlabel("Minute")
    ax2.set_ylabel("Speed (km/h)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # Panel 3: Confidence distribution
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.hist(df["confidence"], bins=20, color="#9b59b6", edgecolor="white")
    ax3.axvline(0.8, color="#e74c3c", linestyle="--", linewidth=1, label="Threshold 0.8")
    ax3.set_title("Classification Confidence")
    ax3.set_xlabel("Confidence")
    ax3.set_ylabel("Frequency")
    ax3.legend(fontsize=8)

    # Panel 4: Vehicle counts by type (stacked area)
    ax4 = fig.add_subplot(2, 3, 4)
    count_cols = [c for c in ["count_car", "count_truck", "count_bus",
                               "count_motorcycle", "count_bicycle"]
                  if c in df.columns]
    if count_cols:
        df_counts = df[count_cols].fillna(0)
        ax4.stackplot(
            x_axis, *[df_counts[c] for c in count_cols],
            labels=[c.replace("count_", "") for c in count_cols],
            colors=[type_colors.get(c.replace("count_", ""), "#999") for c in count_cols],
            alpha=0.8,
        )
        ax4.set_title("Vehicle Counts by Type")
        ax4.set_xlabel("Minute")
        ax4.set_ylabel("Count")
        ax4.legend(loc="upper left", fontsize=7)
    else:
        ax4.text(0.5, 0.5, "No count data", ha="center", va="center")
        ax4.set_title("Vehicle Counts by Type")

    # Panel 5: Speed vs. total vehicles (scatter)
    ax5 = fig.add_subplot(2, 3, 5)
    if "total_vehicles" in df.columns:
        scatter_colors = [colors[c] if c < len(colors) else "#999"
                          for c in df["traffic_state"]]
        ax5.scatter(df["total_vehicles"], df["avg_speed"], c=scatter_colors,
                    alpha=0.7, edgecolors="white", linewidth=0.5)
        ax5.set_xlabel("Total Vehicles")
        ax5.set_ylabel("Avg Speed (km/h)")
        ax5.set_title("Speed vs. Volume")
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, "No volume data", ha="center", va="center")
        ax5.set_title("Speed vs. Volume")

    # Panel 6: Summary text panel
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis("off")
    summary_lines = [
        f"📊 Total records: {len(df)}",
        f"⏱️  Avg speed: {df['avg_speed'].mean():.1f} km/h",
        f"📈 Max speed: {df['avg_speed'].max():.1f} km/h",
        f"📉 Min speed: {df['avg_speed'].min():.1f} km/h",
    ]
    if "total_vehicles" in df.columns:
        summary_lines.append(f"🚗 Total vehicles: {df['total_vehicles'].sum():.0f}")
    if "confidence" in df.columns:
        summary_lines.append(f"🎯 Avg confidence: {df['confidence'].mean():.3f}")
        low_conf = (df["confidence"] < 0.8).sum()
        summary_lines.append(f"⚠️  Low confidence (<0.8): {low_conf}")
    summary_text = "\n".join(summary_lines)
    ax6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment="center",
             fontfamily="monospace", transform=ax6.transAxes)
    ax6.set_title("Summary")

    plt.tight_layout()
    plt.show()


# Execution — requires df_classified from Cell 3
try:
    if df_classified is not None and not df_classified.empty:
        show_dashboard(df_classified)
    else:
        print("⚠️ No classified data — run Cells 2-3 first")
except NameError:
    print("⚠️ df_classified not defined — run Cells 2-3 first")
except Exception as e:
    print(f"🔴 Dashboard error: {e}")